## Application des estimateurs de grande dimensions sur l'optimisation de portefeuille

### DESCRIPTION

L'idée globalement c'est de challenger les estimateurs et d'analyser leur comportement sur des données réelles. Mon objectif est de backtester le portefeuille minimum variance et d'analyser sa variance, son ratio de Sharp ainsi que son comportement sur la frontiere d'efficience.

<small>

## Feuille de route — Application portefeuille : backtest du portefeuille minimum-variance

</small>

<small>

**Paramètres utilisés**

- Fenêtre train : $n_{\text{train}} = 200$ jours de bourse (201 lignes de prix avant `pct_change`)
- Fenêtre test : $n_{\text{test}} = 200$ jours de bourse (même construction)
- Rendements : **simples** (`pct_change`), pas logarithmiques
- Univers de départ : constituants actuels du S&P 500, historique de prix ajustés (`yfinance`)
- Nettoyage : `dropna(axis=1)` strict sur la fenêtre train+test (aucune tolérance de valeur manquante)
- Univers final : restriction aux 420 premiers tickers propres (`p=420`)
- Grille de p : {40, 52, 67, 88, 114, 148, 192, 249, 323, 420}, soit γ = p/200 ∈ {0.20, ..., 2.10}
- 30 tirages aléatoires stratifiés par secteur GICS, par valeur de p
- Méthodes comparées : Sample, Clipping (Marchenko-Pastur), Linear (Ledoit-Wolf 2004), NLS (Ledoit-Wolf non-linéaire 2020), 1/N (benchmark équipondéré)

</small>

<small>

**Phase 1 — Acquisition et préparation des données**

- Constituants S&P 500 (`sp500_constituents.csv`) avec colonne GICS Sector.
- Téléchargement `yfinance` sur 30 ans, prix ajustés (`auto_adjust=True`), tous les tickers de l'indice.
- Sélection des `N_train + N_test` dernières lignes de prix, puis `dropna(axis=1)` : seuls les tickers sans aucune valeur manquante sur cette fenêtre récente sont conservés.
- Restriction aux 420 premiers tickers propres pour fixer l'univers maximal.
- Calcul des rendements simples (`pct_change`) séparément sur train et test.

</small>

<small>

**Phase 2 — Échantillonnage stratifié des univers par γ**

- Pour chaque p de la grille : proportions sectorielles calculées sur les tickers propres, arrondies puis corrigées pour sommer exactement à p.
- 30 tirages indépendants par p : échantillon aléatoire sans remise par secteur, puis concaténation.
- Sécurité : un secteur ne peut fournir plus de tickers qu'il n'en contient (`clip`).

</small>

<small>

**Phase 3 — Calcul des estimateurs et des portefeuilles GMV (Global Minimum Variance)**

- Pour chaque (p, tirage) : normalisation des rendements train, calcul de la matrice de corrélation empirique et de sa décomposition spectrale.
- 4 estimateurs appliqués sur la corrélation (Sample, Clipping, Linear, NLS), puis reconstruction en covariance via $D\cdot\hat\Sigma_{\text{corr}}\cdot D$ (écarts-types réels).
- Portefeuille GMV par méthode : $w = \hat\Sigma_{\text{cov}}^{-1}\mathbf{1}/(\mathbf{1}^\top\hat\Sigma_{\text{cov}}^{-1}\mathbf{1})$, résolu via `np.linalg.solve`.
- 1/N ajouté comme 5ᵉ méthode (poids égaux ; réutilise la matrice de covariance de Sample pour le calcul de variance).
- Calcul parallélisé sur les 300 combinaisons (p × tirage) via `joblib.Parallel`.

</small>

<small>

**Phase 4 — Métriques in-sample et out-of-sample**

- Pour chaque (p, tirage, méthode) : variance in-sample prédite ($w^\top\hat\Sigma w$), rendement moyen et Sharpe in-sample (rendement réalisé sur train / racine de la variance prédite).
- Même calcul out-of-sample : poids figés (calculés sur train) appliqués aux rendements test, variance et rendement réellement réalisés.
- Écarts (out − in) de variance et de Sharpe stockés pour chaque ligne; signature empirique de l'« error maximization » de Michaud (1989).
- Résultat centralisé dans `df_resultats`.

</small>

<small>

**Phase 5 — Visualisation**

- Courbes de moyenne par méthode, en fonction de γ : variance in-sample, variance out-of-sample, écart de variance, Sharpe in-sample, Sharpe out-of-sample, écart de Sharpe.
- Rang moyen par méthode (1 = meilleur Sharpe réalisé), agrégé sur les 30 tirages, affiché en heatmap p × méthode.
- Taux de victoire par méthode (fraction des tirages où elle obtient le meilleur Sharpe réalisé), même format de heatmap.
- Illustration sur la frontière d'efficience, au ratio de concentration le plus élevé (p=420) : reconstruction de la frontière in-sample pour Clipping/Linear/NLS, avec le point in-sample (sommet, exact par construction) et le point out-of-sample (réalisé) pour chaque méthode, plus 1/N. Une seconde version ajoute Sample via `pinv()` (la matrice étant singulière à ce p), avec un axe recadré pour rester lisible.

</small>

<small>

<!-- **Non encore implémenté dans ce notebook** (prévu dans la feuille de route initiale)

- Test de significativité de Jobson-Korkie (correction de Memmel, 1998) entre méthodes.
- Extension walk-forward glissante avec rebalancement périodique et suivi du turnover. -->

</small>


<small>

## Hypothèses de l'application portefeuille

</small>

<small>

**1. Hypothèses de stationnarité**

- **Stationnarité de $\hat{\bm\Sigma}$** : on suppose que la structure de covariance estimée sur train reste valable sur test : pas de rupture structurelle, pas de changement de régime de marché entre les deux fenêtres. C'est l'hypothèse la plus centrale de toute la démarche : sans elle, comparer performance in-sample et out-of-sample ne mesurerait rien d'interprétable, puisqu'on comparerait deux mondes différents plutôt que la qualité d'un estimateur.
- **Stationnarité de $\hat{\bm\mu}$** : implicitement supposée aussi, bien que le portefeuille GMV (Global Minimum Variance) ne l'utilise pas pour construire les poids. Elle intervient quand même dès qu'on interprète `mean_out` ou le Sharpe réalisé.
- **Observations i.i.d. (indépendantes et identiquement distribuées) dans le temps** : toute la théorie RMT (Random Matrix Theory) sous-jacente suppose que les $n$ observations temporelles sont i.i.d. Or les rendements financiers réels présentent de l'autocorrélation et du regroupement de volatilité.

</small>

<small>

**2. Hypothèses liées à la construction de $\hat{\bm\Sigma}_{\text{cov}}$**

- **Séparabilité corrélation/volatilité** : la reconstruction $D\cdot\hat{\bm\Sigma}_{\text{corr}}\cdot D$ suppose qu'on peut estimer séparément la structure de corrélation et les écarts-types individuels, puis les recombiner sans perte. Ça suppose implicitement que $D$ (les écarts-types réalisés sur train) reste lui aussi représentatif de la période test donc une deuxième couche de l'hypothèse de stationnarité, appliquée cette fois spécifiquement aux volatilités individuelles plutôt qu'à la corrélation.

</small>

<small>

**3. Une hypothèse technique souvent négligée : l'additivité des rendements logarithmiques**

Un rendement de portefeuille est, par définition, une combinaison linéaire des rendements **simples** des actifs : $R_p = \sum_i w_i R_i$. Mais on a travaillé tout du long avec des rendements **logarithmiques**, $r_i = \log(1+R_i)$, et calculé $\bm w^\top \bm r$ comme si c'était le rendement du portefeuille. Or $\log(1+\sum_i w_i R_i) \neq \sum_i w_i \log(1+R_i)$ en général — l'égalité n'est qu'une **approximation au premier ordre**, valable quand les rendements individuels sont petits. C'est une approximation standard et généralement sans grande conséquence à l'échelle quotidienne, mais c'est une hypothèse implicite qu'on n'avait jamais rendue explicite.

</small>

<small>

**4. Biais déjà identifiés**

- Biais de survivance (univers actuel projeté sur 30 ans).
- Nettoyage strict des tickers (`dropna` sans tolérance), qui suppose que l'exclusion n'est pas corrélée avec le phénomène étudié.

</small>

In [ ]:
%run setup.py

#### Phase 7.1 : Acquisition et préparation des données

In [ ]:
import itertools
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from joblib import Parallel, delayed

from models.adapted_clipping import *
from models.base import *
from models.data import *
from models.linear_shrinkage import *
from models.non_linear_shrinkage import *
from models.oracle import *
from models.rotation_invariant import *
from models.sample_correlation import *
from models.silverstein_solver import *
from models.simple_clipping import *
from tools.simulation_tools import *
from tools.tools import *

In [ ]:
# ============================================================
# PARAMÈTRES
# ============================================================
ANNEES      = 30
N_ACTIONS   = 300
SEED        = 42
SEUIL_NAN   = 0.10
# ============================================================

In [ ]:
# import pandas as pd

# constituents = pd.read_csv(
#     "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv"
# )
# constituents["Symbol"] = constituents["Symbol"].str.replace(".", "-", regex=False)

# print(f"{len(constituents)} entreprises trouvées")
# print(constituents[["Symbol", "Security", "GICS Sector"]].head())

In [ ]:
# constituents.to_csv("sp500_constituents.csv", index=False)

In [ ]:
current_dir = Path.cwd().parent
file_path = current_dir / "data" / "raw" / "sp500_constituents.csv"

sp500_constituants = pd.read_csv(file_path, sep=',')
sp500_constituants

In [ ]:
end_date = '2026-07-01'# datetime.today().strftime("%Y-%m-%d")
start_date = (datetime(2026, 7, 1) - timedelta(days=365 * ANNEES)).strftime("%Y-%m-%d")

tous_les_tickers = sp500_constituants["Symbol"].tolist()

raw = yf.download(
    tous_les_tickers,
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=True,
)["Close"]

print(f"Téléchargement terminé : {raw.shape[0]} jours, {raw.shape[1]} tickers")

In [ ]:
raw.to_csv("sp500_raw_data.csv", index=True)

In [ ]:
sp500_raw_data = pd.read_csv('sp500_raw_data.csv', sep=',', index_col='Date')
sp500_raw_data

In [ ]:
N_train = 200 + 1
N_test = 200 + 1
p = 420
SEUIL_NAN = 0

sp500_train_test_raw_data = sp500_raw_data.iloc[-(N_train + N_test):,:]
sp500_train_test_raw_data

In [ ]:
sp500_train_test_clean_data = sp500_train_test_raw_data.dropna(axis=1)
sp500_train_test_clean_data

In [ ]:
prices_train = sp500_train_test_clean_data.iloc[:N_train,:p]
prices_test = sp500_train_test_clean_data.iloc[N_train:N_train + N_test,:p]
prices_train

In [ ]:
prices_test

In [ ]:
returns_train = prices_train.pct_change().dropna(axis=0)
returns_test = prices_test.pct_change().dropna(axis=0)
returns_train

In [ ]:
returns_test

In [ ]:
train = returns_train
test = returns_test

### Phase 2 : Échantillonnage stratifié des univers par γ

In [ ]:
p_grid = [40, 52, 67, 88, 114, 148, 192, 249, 323, 412]
n_draws = 30
rng = np.random.default_rng(seed=42)

# Sous-étape 1 : ne garder que les secteurs des tickers propres
clean_sectors = sp500_constituants[sp500_constituants["Symbol"].isin(train.columns)]
sector_counts = clean_sectors["GICS Sector"].value_counts()
proportions = sector_counts / sector_counts.sum()

tirages = {}

for p in p_grid:
    # Sous-étape 2 : nombre de tickers par secteur, arrondi, résidu corrigé sur le plus grand secteur
    counts_per_sector = (proportions * p).round().astype(int)
    ecart = p - counts_per_sector.sum()
    secteur_principal = counts_per_sector.idxmax()
    counts_per_sector[secteur_principal] += ecart

    # Sécurité : un secteur ne peut pas fournir plus de tickers qu'il n'en contient
    counts_per_sector = counts_per_sector.clip(upper=sector_counts)

    # Sous-étape 3 et 4 : 30 tirages indépendants, un par secteur puis concaténation
    tirages[p] = []
    for _ in range(n_draws):
        selection = []
        for secteur, k in counts_per_sector.items():
            pool = clean_sectors.loc[clean_sectors["GICS Sector"] == secteur, "Symbol"]
            tirage_secteur = pool.sample(n=k, replace=False, random_state=rng)
            selection.extend(tirage_secteur.tolist())
        tirages[p].append(selection)

#### Phase 3 : Calcul des estimateurs et des portefeuilles GMV (Global Minimum Variance)

In [ ]:


p_grid = [40, 52, 67, 88, 114, 148, 192, 249, 323, 412]
n_train = 200
n_draws = 30


def run_tirage_p3(p, i):
    tirage = tirages[p][i]
    sub = train[tirage]

    std_vec = sub.std(axis=0).values
    D = np.diag(std_vec)

    data_object = DataClass().fit(sub.values)
    # X_norm = normalisation(sub.values)
    S_corr = SampleCorrelationEstimator().fit(data_object).correlation_

    matrices_corr = {
        "Sample": S_corr,
        "Clipping": NaiveClippingEstimator().fit(data_object).correlation_,
        "Linear": LinearShrinkageEstimator().fit(data_object).correlation_,
        "NLS": NonLinearShrinkageEstimator().fit(data_object).correlation_,
    }

    portefeuilles = {}
    for nom, S_c in matrices_corr.items():
        sigma_cov = D @ S_c @ D
        w = np.linalg.solve(sigma_cov, np.ones(p))
        w = w / w.sum()
        portefeuilles[nom] = {"w": w, "sigma_cov": sigma_cov}

    w_equal = np.ones(p) / p
    portefeuilles["1/N"] = {"w": w_equal, "sigma_cov": portefeuilles["Sample"]["sigma_cov"]}

    return p, i, portefeuilles


taches = list(itertools.product(p_grid, range(n_draws)))
print(f"{len(taches)} taches a lancer")

sorties = Parallel(n_jobs=-2, verbose=5)(
    delayed(run_tirage_p3)(p, i) for (p, i) in taches
)

resultats = {p: [None] * n_draws for p in p_grid}
for p, i, portefeuilles in sorties:
    resultats[p][i] = portefeuilles

#### Phase 4

In [ ]:
lignes = []

for p in p_grid:
    for i in range(n_draws):
        tirage = tirages[p][i]
        sub_train = train[tirage].values
        sub_test = test[tirage].values

        for nom, contenu in resultats[p][i].items():
            w = contenu["w"]
            sigma_cov = contenu["sigma_cov"]

            var_in = w @ sigma_cov @ w
            rendements_in = sub_train @ w
            mean_in = rendements_in.mean()
            sharpe_in = mean_in / np.sqrt(var_in)

            rendements_out = sub_test @ w
            var_out = rendements_out.var(ddof=1)
            mean_out = rendements_out.mean()
            sharpe_out = mean_out / np.sqrt(var_out)

            lignes.append({
                "p": p,
                "tirage": i,
                "methode": nom,
                "var_in": var_in,
                "var_out": var_out,
                "mean_in": mean_in,
                "mean_out": mean_out,
                "sharpe_in": sharpe_in,
                "sharpe_out": sharpe_out,
                "ecart_variance": var_out - var_in,
                "ecart_sharpe": sharpe_out - sharpe_in,
            })

df_resultats = pd.DataFrame(lignes)

In [ ]:
df_resultats

#### Phase 5 : Visualisation

Variance — in-sample

In [ ]:
couleurs = {"Sample": "black", "Clipping": "royalblue", "Linear": "forestgreen", "NLS": "red", "1/N": "purple"}
methodes = list(couleurs.keys())
offsets = np.linspace(-0.3, 0.3, len(methodes))

In [ ]:
n_train = 200
gamma_values = [p / n_train for p in p_grid]
positions = np.arange(len(p_grid))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for methode, couleur in couleurs.items():
    moyennes = df_resultats[df_resultats["methode"] == methode].groupby("p")["var_in"].mean()
    ax.plot(positions, moyennes.reindex(p_grid).values, marker="o", linewidth=2, color=couleur, label=methode)

ax.set_xticks(positions)
ax.set_xticklabels([f"{g:.2f}" for g in gamma_values])
ax.set_xlabel("Ratio de concentration (gamma)")
ax.set_ylabel("Variance (in-sample, predite)")
ax.legend()
# ax.set_ylim(0, 0.00006)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

Variance — out-of-sample

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for methode, couleur in couleurs.items():
    moyennes = df_resultats[df_resultats["methode"] == methode].groupby("p")["var_out"].mean()
    ax.plot(positions, moyennes.reindex(p_grid).values, marker="o", linewidth=2, color=couleur, label=methode)

ax.set_xticks(positions)
ax.set_xticklabels([f"{g:.2f}" for g in gamma_values])
ax.set_xlabel("Ratio de concentration (gamma)")
ax.set_ylabel("Variance (out-of-sample)")
ax.legend()
ax.set_ylim(0, 0.0002)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

Variance : écart (out − in)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for methode, couleur in couleurs.items():
    moyennes = df_resultats[df_resultats["methode"] == methode].groupby("p")["ecart_variance"].mean()
    ax.plot(positions, moyennes.reindex(p_grid).values, marker="o", linewidth=2, color=couleur, label=methode)

ax.set_xticks(positions)
ax.set_xticklabels([f"{g:.2f}" for g in gamma_values])
ax.set_xlabel("Ratio de concentration (gamma)")
ax.set_ylabel("Ecart variance (out - in)")
ax.legend()
ax.set_ylim(0, 0.0002)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

Sharpe — in-sample

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for methode, couleur in couleurs.items():
    moyennes = df_resultats[df_resultats["methode"] == methode].groupby("p")["sharpe_in"].mean()
    ax.plot(positions, moyennes.reindex(p_grid).values, marker="o", linewidth=2, color=couleur, label=methode)

ax.set_xticks(positions)
ax.set_xticklabels([f"{g:.2f}" for g in gamma_values])
ax.set_xlabel("Ratio de concentration (gamma)")
ax.set_ylabel("Ratio de Sharpe (in-sample)")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for methode, couleur in couleurs.items():
    moyennes = df_resultats[df_resultats["methode"] == methode].groupby("p")["sharpe_in"].mean()
    ax.plot(positions, moyennes.reindex(p_grid).values, marker="o", linewidth=2, color=couleur, label=methode)

ax.set_xticks(positions)
ax.set_xticklabels([f"{g:.2f}" for g in gamma_values])
ax.set_xlabel("Ratio de concentration (gamma)")
ax.set_ylabel("Ratio de Sharpe (in-sample)")
ax.legend()
ax.set_ylim(-0.1, 0.5)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

Sharpe — out-of-sample

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for methode, couleur in couleurs.items():
    moyennes = df_resultats[df_resultats["methode"] == methode].groupby("p")["sharpe_out"].mean()
    ax.plot(positions, moyennes.reindex(p_grid).values, marker="o", linewidth=2, color=couleur, label=methode)

ax.set_xticks(positions)
ax.set_xticklabels([f"{g:.2f}" for g in gamma_values])
ax.set_xlabel("Ratio de concentration (gamma)")
ax.set_ylabel("Ratio de Sharpe (out-of-sample)")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

Sharpe — écart (out − in)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for methode, couleur in couleurs.items():
    moyennes = df_resultats[df_resultats["methode"] == methode].groupby("p")["ecart_sharpe"].mean()
    ax.plot(positions, moyennes.reindex(p_grid).values, marker="o", linewidth=2, color=couleur, label=methode)

ax.set_xticks(positions)
ax.set_xticklabels([f"{g:.2f}" for g in gamma_values])
ax.set_xlabel("Ratio de concentration (gamma)")
ax.set_ylabel("Ecart Sharpe (out - in)")
ax.legend()
ax.set_ylim(-0.5, 0.5)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
df_resultats["rang"] = df_resultats.groupby(["p", "tirage"])["sharpe_out"].rank(ascending=False, method="first")

rang_moyen = df_resultats.groupby(["p", "methode"])["rang"].mean()
taux_victoire = df_resultats.groupby(["p", "methode"])["rang"].apply(lambda x: (x == 1).mean())

Rang moyen par méthode vs γ

Question : quelle méthode se classe le mieux en moyenne, indépendamment de l'ampleur des écarts (donc insensible à l'explosion de Sample) ? Insight : si Sample a un rang moyen qui se dégrade nettement à mesure que γ augmente (au-delà de simplement "5e systématiquement"), ça confirme le classement qu'on voit déjà sur le Sharpe brut, mais de façon plus robuste statistiquement.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for methode, couleur in couleurs.items():
    valeurs = rang_moyen.xs(methode, level="methode").reindex(p_grid).values
    ax.plot(positions, valeurs, marker="o", linewidth=2, color=couleur, label=methode)

ax.set_xticks(positions)
ax.set_xticklabels([f"{g:.2f}" for g in gamma_values])
ax.set_xlabel("Ratio de concentration (gamma)")
ax.set_ylabel("Rang moyen (1 = meilleur Sharpe out-of-sample)")
ax.invert_yaxis()
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
methodes = list(couleurs.keys())
n_methodes = len(methodes)

rang_moyen_matrix = np.array([[rang_moyen.loc[(p, m)] for p in p_grid] for m in methodes])

plt.figure(figsize=(11, 4.5))
im = plt.imshow(rang_moyen_matrix, aspect="auto", cmap="RdYlGn_r", vmin=1, vmax=n_methodes)
plt.yticks(range(n_methodes), methodes)
plt.xticks(range(len(gamma_values)), [f"{g:.2g}" for g in gamma_values], rotation=45)
plt.xlabel("Ratio de concentration (gamma)")
plt.title("Rang moyen de chaque methode (1=meilleur) a chaque gamma")
for i in range(n_methodes):
    for j in range(len(gamma_values)):
        plt.text(j, i, f"{rang_moyen_matrix[i,j]:.1f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="Rang moyen")
plt.tight_layout()
plt.show()

Taux de victoire par méthode vs γ

Question : quelle fraction des 30 tirages chaque méthode remporte-t-elle (meilleur Sharpe réalisé) ? Insight : complémentaire au rang moyen, un taux de victoire qui grimpe avec γ pour NLS/Linear confirmerait qu'elles deviennent systématiquement les meilleures, pas juste "en moyenne meilleures à cause de quelques tirages extrêmes".

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for methode, couleur in couleurs.items():
    valeurs = taux_victoire.xs(methode, level="methode").reindex(p_grid).values
    ax.plot(positions, valeurs, marker="o", linewidth=2, color=couleur, label=methode)

ax.set_xticks(positions)
ax.set_xticklabels([f"{g:.2f}" for g in gamma_values])
ax.set_xlabel("Ratio de concentration (gamma)")
ax.set_ylabel("Taux de victoire (meilleur Sharpe out-of-sample)")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
taux_victoire_matrix = np.array([[taux_victoire.loc[(p, m)] for p in p_grid] for m in methodes])

plt.figure(figsize=(11, 4.5))
im = plt.imshow(taux_victoire_matrix, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
plt.yticks(range(n_methodes), methodes)
plt.xticks(range(len(gamma_values)), [f"{g:.2g}" for g in gamma_values], rotation=45)
plt.xlabel("Ratio de concentration (gamma)")
plt.title("Taux de victoire de chaque methode (meilleur Sharpe out-of-sample) a chaque gamma")
for i in range(n_methodes):
    for j in range(len(gamma_values)):
        plt.text(j, i, f"{taux_victoire_matrix[i,j]:.2f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, label="Taux de victoire")
plt.tight_layout()
plt.show()

#### Frontiere d'efficience

In [ ]:
p_max = 412
tirage_max = tirages[p_max][0]
mu_hat = train[tirage_max].mean(axis=0).values

fig, ax = plt.subplots(figsize=(9, 6))

# --- Clipping ---
sigma_c = resultats[p_max][0]["Clipping"]["sigma_cov"]
x1 = np.linalg.solve(sigma_c, np.ones(p_max))
x2 = np.linalg.solve(sigma_c, mu_hat)
A = np.ones(p_max) @ x1
B = np.ones(p_max) @ x2
C = mu_hat @ x2
D = A * C - B**2
mu_grid = np.linspace(mu_hat.min(), mu_hat.max(), 200)
sigma_grid = np.sqrt(np.maximum((A * mu_grid**2 - 2 * B * mu_grid + C) / D, 0))
ax.plot(sigma_grid, mu_grid, color="royalblue", linewidth=1.5, alpha=0.7, label="Frontiere Clipping (in-sample)")
sigma_in = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Clipping")]["var_in"].values[0])
mean_in = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Clipping")]["mean_in"].values[0]
sigma_out = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Clipping")]["var_out"].values[0])
mean_out = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Clipping")]["mean_out"].values[0]
ax.scatter([sigma_in], [mean_in], color="royalblue", marker="*", s=200, zorder=5, label="Clipping in-sample")
ax.scatter([sigma_out], [mean_out], color="royalblue", marker="X", s=120, zorder=5, label="Clipping out-of-sample")
ax.plot([sigma_in, sigma_out], [mean_in, mean_out], color="royalblue", linestyle="--", linewidth=1, alpha=0.6)

# --- Linear ---
sigma_c = resultats[p_max][0]["Linear"]["sigma_cov"]
x1 = np.linalg.solve(sigma_c, np.ones(p_max))
x2 = np.linalg.solve(sigma_c, mu_hat)
A = np.ones(p_max) @ x1
B = np.ones(p_max) @ x2
C = mu_hat @ x2
D = A * C - B**2
sigma_grid = np.sqrt(np.maximum((A * mu_grid**2 - 2 * B * mu_grid + C) / D, 0))
ax.plot(sigma_grid, mu_grid, color="forestgreen", linewidth=1.5, alpha=0.7, label="Frontiere Linear (in-sample)")
sigma_in = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Linear")]["var_in"].values[0])
mean_in = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Linear")]["mean_in"].values[0]
sigma_out = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Linear")]["var_out"].values[0])
mean_out = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Linear")]["mean_out"].values[0]
ax.scatter([sigma_in], [mean_in], color="forestgreen", marker="*", s=200, zorder=5, label="Linear in-sample")
ax.scatter([sigma_out], [mean_out], color="forestgreen", marker="X", s=120, zorder=5, label="Linear out-of-sample")
ax.plot([sigma_in, sigma_out], [mean_in, mean_out], color="forestgreen", linestyle="--", linewidth=1, alpha=0.6)

# --- NLS ---
sigma_c = resultats[p_max][0]["NLS"]["sigma_cov"]
x1 = np.linalg.solve(sigma_c, np.ones(p_max))
x2 = np.linalg.solve(sigma_c, mu_hat)
A = np.ones(p_max) @ x1
B = np.ones(p_max) @ x2
C = mu_hat @ x2
D = A * C - B**2
sigma_grid = np.sqrt(np.maximum((A * mu_grid**2 - 2 * B * mu_grid + C) / D, 0))
ax.plot(sigma_grid, mu_grid, color="red", linewidth=1.5, alpha=0.7, label="Frontiere NLS (in-sample)")
sigma_in = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="NLS")]["var_in"].values[0])
mean_in = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="NLS")]["mean_in"].values[0]
sigma_out = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="NLS")]["var_out"].values[0])
mean_out = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="NLS")]["mean_out"].values[0]
ax.scatter([sigma_in], [mean_in], color="red", marker="*", s=200, zorder=5, label="NLS in-sample")
ax.scatter([sigma_out], [mean_out], color="red", marker="X", s=120, zorder=5, label="NLS out-of-sample")
ax.plot([sigma_in, sigma_out], [mean_in, mean_out], color="red", linestyle="--", linewidth=1, alpha=0.6)

# --- 1/N (pas de frontiere, juste les deux points) ---
sigma_in = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="1/N")]["var_in"].values[0])
mean_in = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="1/N")]["mean_in"].values[0]
sigma_out = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="1/N")]["var_out"].values[0])
mean_out = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="1/N")]["mean_out"].values[0]
ax.scatter([sigma_in], [mean_in], color="gray", marker="*", s=200, zorder=5, label="1/N in-sample")
ax.scatter([sigma_out], [mean_out], color="gray", marker="X", s=120, zorder=5, label="1/N out-of-sample")
ax.plot([sigma_in, sigma_out], [mean_in, mean_out], color="gray", linestyle="--", linewidth=1, alpha=0.6)

ax.set_xlabel("Risque (ecart-type)")
ax.set_ylabel("Rendement moyen")
ax.set_title(f"Portefeuilles sur la frontiere efficiente, gamma={p_max/200:.2f}")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
p_max = 412
tirage_max = tirages[p_max][0]
mu_hat = train[tirage_max].mean(axis=0).values

fig, ax = plt.subplots(figsize=(9, 6))

# --- Sample (matrice singuliere a p=420 : pinv() au lieu de solve(), repli numerique pour l'affichage uniquement) ---
sigma_c = resultats[p_max][0]["Sample"]["sigma_cov"]
sigma_pinv = np.linalg.pinv(sigma_c, rcond=1e-10)
x1 = sigma_pinv @ np.ones(p_max)
x2 = sigma_pinv @ mu_hat
A = np.ones(p_max) @ x1
B = np.ones(p_max) @ x2
C = mu_hat @ x2
D = A * C - B**2
mu_grid = np.linspace(mu_hat.min(), mu_hat.max(), 200)
sigma_grid = np.sqrt(np.maximum((A * mu_grid**2 - 2 * B * mu_grid + C) / D, 0))
ax.plot(sigma_grid, mu_grid, color="black", linewidth=1.5, alpha=0.7, linestyle=":", label="Frontiere Sample (in-sample, pinv)")
sigma_in = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Sample")]["var_in"].values[0])
mean_in = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Sample")]["mean_in"].values[0]
sigma_out = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Sample")]["var_out"].values[0])
mean_out = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Sample")]["mean_out"].values[0]
ax.scatter([sigma_in], [mean_in], color="black", marker="*", s=200, zorder=5, label="Sample in-sample")
ax.scatter([sigma_out], [mean_out], color="black", marker="X", s=120, zorder=5, label="Sample out-of-sample")
ax.plot([sigma_in, sigma_out], [mean_in, mean_out], color="black", linestyle="--", linewidth=1, alpha=0.6)

# --- Clipping ---
sigma_c = resultats[p_max][0]["Clipping"]["sigma_cov"]
x1 = np.linalg.solve(sigma_c, np.ones(p_max))
x2 = np.linalg.solve(sigma_c, mu_hat)
A = np.ones(p_max) @ x1
B = np.ones(p_max) @ x2
C = mu_hat @ x2
D = A * C - B**2
sigma_grid = np.sqrt(np.maximum((A * mu_grid**2 - 2 * B * mu_grid + C) / D, 0))
ax.plot(sigma_grid, mu_grid, color="royalblue", linewidth=1.5, alpha=0.7, label="Frontiere Clipping (in-sample)")
sigma_in = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Clipping")]["var_in"].values[0])
mean_in = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Clipping")]["mean_in"].values[0]
sigma_out = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Clipping")]["var_out"].values[0])
mean_out = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Clipping")]["mean_out"].values[0]
ax.scatter([sigma_in], [mean_in], color="royalblue", marker="*", s=200, zorder=5, label="Clipping in-sample")
ax.scatter([sigma_out], [mean_out], color="royalblue", marker="X", s=120, zorder=5, label="Clipping out-of-sample")
ax.plot([sigma_in, sigma_out], [mean_in, mean_out], color="royalblue", linestyle="--", linewidth=1, alpha=0.6)

# --- Linear ---
sigma_c = resultats[p_max][0]["Linear"]["sigma_cov"]
x1 = np.linalg.solve(sigma_c, np.ones(p_max))
x2 = np.linalg.solve(sigma_c, mu_hat)
A = np.ones(p_max) @ x1
B = np.ones(p_max) @ x2
C = mu_hat @ x2
D = A * C - B**2
sigma_grid = np.sqrt(np.maximum((A * mu_grid**2 - 2 * B * mu_grid + C) / D, 0))
ax.plot(sigma_grid, mu_grid, color="forestgreen", linewidth=1.5, alpha=0.7, label="Frontiere Linear (in-sample)")
sigma_in = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Linear")]["var_in"].values[0])
mean_in = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Linear")]["mean_in"].values[0]
sigma_out = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Linear")]["var_out"].values[0])
mean_out = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Linear")]["mean_out"].values[0]
ax.scatter([sigma_in], [mean_in], color="forestgreen", marker="*", s=200, zorder=5, label="Linear in-sample")
ax.scatter([sigma_out], [mean_out], color="forestgreen", marker="X", s=120, zorder=5, label="Linear out-of-sample")
ax.plot([sigma_in, sigma_out], [mean_in, mean_out], color="forestgreen", linestyle="--", linewidth=1, alpha=0.6)

# --- NLS ---
sigma_c = resultats[p_max][0]["NLS"]["sigma_cov"]
x1 = np.linalg.solve(sigma_c, np.ones(p_max))
x2 = np.linalg.solve(sigma_c, mu_hat)
A = np.ones(p_max) @ x1
B = np.ones(p_max) @ x2
C = mu_hat @ x2
D = A * C - B**2
sigma_grid = np.sqrt(np.maximum((A * mu_grid**2 - 2 * B * mu_grid + C) / D, 0))
ax.plot(sigma_grid, mu_grid, color="red", linewidth=1.5, alpha=0.7, label="Frontiere NLS (in-sample)")
sigma_in = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="NLS")]["var_in"].values[0])
mean_in = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="NLS")]["mean_in"].values[0]
sigma_out = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="NLS")]["var_out"].values[0])
mean_out = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="NLS")]["mean_out"].values[0]
ax.scatter([sigma_in], [mean_in], color="red", marker="*", s=200, zorder=5, label="NLS in-sample")
ax.scatter([sigma_out], [mean_out], color="red", marker="X", s=120, zorder=5, label="NLS out-of-sample")
ax.plot([sigma_in, sigma_out], [mean_in, mean_out], color="red", linestyle="--", linewidth=1, alpha=0.6)

# --- 1/N (pas de frontiere, juste les deux points) ---
sigma_in = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="1/N")]["var_in"].values[0])
mean_in = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="1/N")]["mean_in"].values[0]
sigma_out = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="1/N")]["var_out"].values[0])
mean_out = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="1/N")]["mean_out"].values[0]
ax.scatter([sigma_in], [mean_in], color="gray", marker="*", s=200, zorder=5, label="1/N in-sample")
ax.scatter([sigma_out], [mean_out], color="gray", marker="X", s=120, zorder=5, label="1/N out-of-sample")
ax.plot([sigma_in, sigma_out], [mean_in, mean_out], color="gray", linestyle="--", linewidth=1, alpha=0.6)

ax.set_xlabel("Risque (ecart-type)")
ax.set_xlim(0, 0.01)
ax.set_ylabel("Rendement moyen")
ax.set_title(f"Portefeuilles sur la frontiere efficiente, gamma={p_max/200:.2f} (Sample: frontiere via pinv, a interpreter avec prudence)")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# p_grid = [40, 52, 67, 88, 114, 148, 192, 249, 323, 412]

p_max = 40 
tirage_max = tirages[p_max][0]
mu_hat = train[tirage_max].mean(axis=0).values

fig, ax = plt.subplots(figsize=(9, 6))

# --- Clipping ---
sigma_c = resultats[p_max][0]["Clipping"]["sigma_cov"]
x1 = np.linalg.solve(sigma_c, np.ones(p_max))
x2 = np.linalg.solve(sigma_c, mu_hat)
A = np.ones(p_max) @ x1
B = np.ones(p_max) @ x2
C = mu_hat @ x2
D = A * C - B**2
mu_grid = np.linspace(mu_hat.min(), mu_hat.max(), 200)
sigma_grid = np.sqrt(np.maximum((A * mu_grid**2 - 2 * B * mu_grid + C) / D, 0))
ax.plot(sigma_grid, mu_grid, color="royalblue", linewidth=1.5, alpha=0.7, label="Frontiere Clipping (in-sample)")
sigma_in = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Clipping")]["var_in"].values[0])
mean_in = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Clipping")]["mean_in"].values[0]
sigma_out = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Clipping")]["var_out"].values[0])
mean_out = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Clipping")]["mean_out"].values[0]
ax.scatter([sigma_in], [mean_in], color="royalblue", marker="*", s=200, zorder=5, label="Clipping in-sample")
ax.scatter([sigma_out], [mean_out], color="royalblue", marker="X", s=120, zorder=5, label="Clipping out-of-sample")
ax.plot([sigma_in, sigma_out], [mean_in, mean_out], color="royalblue", linestyle="--", linewidth=1, alpha=0.6)

# --- Linear ---
sigma_c = resultats[p_max][0]["Linear"]["sigma_cov"]
x1 = np.linalg.solve(sigma_c, np.ones(p_max))
x2 = np.linalg.solve(sigma_c, mu_hat)
A = np.ones(p_max) @ x1
B = np.ones(p_max) @ x2
C = mu_hat @ x2
D = A * C - B**2
sigma_grid = np.sqrt(np.maximum((A * mu_grid**2 - 2 * B * mu_grid + C) / D, 0))
ax.plot(sigma_grid, mu_grid, color="forestgreen", linewidth=1.5, alpha=0.7, label="Frontiere Linear (in-sample)")
sigma_in = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Linear")]["var_in"].values[0])
mean_in = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Linear")]["mean_in"].values[0]
sigma_out = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Linear")]["var_out"].values[0])
mean_out = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Linear")]["mean_out"].values[0]
ax.scatter([sigma_in], [mean_in], color="forestgreen", marker="*", s=200, zorder=5, label="Linear in-sample")
ax.scatter([sigma_out], [mean_out], color="forestgreen", marker="X", s=120, zorder=5, label="Linear out-of-sample")
ax.plot([sigma_in, sigma_out], [mean_in, mean_out], color="forestgreen", linestyle="--", linewidth=1, alpha=0.6)

# --- NLS ---
sigma_c = resultats[p_max][0]["NLS"]["sigma_cov"]
x1 = np.linalg.solve(sigma_c, np.ones(p_max))
x2 = np.linalg.solve(sigma_c, mu_hat)
A = np.ones(p_max) @ x1
B = np.ones(p_max) @ x2
C = mu_hat @ x2
D = A * C - B**2
sigma_grid = np.sqrt(np.maximum((A * mu_grid**2 - 2 * B * mu_grid + C) / D, 0))
ax.plot(sigma_grid, mu_grid, color="red", linewidth=1.5, alpha=0.7, label="Frontiere NLS (in-sample)")
sigma_in = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="NLS")]["var_in"].values[0])
mean_in = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="NLS")]["mean_in"].values[0]
sigma_out = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="NLS")]["var_out"].values[0])
mean_out = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="NLS")]["mean_out"].values[0]
ax.scatter([sigma_in], [mean_in], color="red", marker="*", s=200, zorder=5, label="NLS in-sample")
ax.scatter([sigma_out], [mean_out], color="red", marker="X", s=120, zorder=5, label="NLS out-of-sample")
ax.plot([sigma_in, sigma_out], [mean_in, mean_out], color="red", linestyle="--", linewidth=1, alpha=0.6)

# --- 1/N (pas de frontiere, juste les deux points) ---
sigma_in = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="1/N")]["var_in"].values[0])
mean_in = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="1/N")]["mean_in"].values[0]
sigma_out = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="1/N")]["var_out"].values[0])
mean_out = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="1/N")]["mean_out"].values[0]
ax.scatter([sigma_in], [mean_in], color="gray", marker="*", s=200, zorder=5, label="1/N in-sample")
ax.scatter([sigma_out], [mean_out], color="gray", marker="X", s=120, zorder=5, label="1/N out-of-sample")
ax.plot([sigma_in, sigma_out], [mean_in, mean_out], color="gray", linestyle="--", linewidth=1, alpha=0.6)

ax.set_xlabel("Risque (ecart-type)")
ax.set_ylabel("Rendement moyen")
ax.set_title(f"Portefeuilles sur la frontiere efficiente, gamma={p_max/200:.2f}")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
p_max = 52
tirage_max = tirages[p_max][0]
mu_hat = train[tirage_max].mean(axis=0).values

fig, ax = plt.subplots(figsize=(9, 6))

# --- Sample (matrice singuliere a p=420 : pinv() au lieu de solve(), repli numerique pour l'affichage uniquement) ---
sigma_c = resultats[p_max][0]["Sample"]["sigma_cov"]
sigma_pinv = np.linalg.pinv(sigma_c, rcond=1e-10)
x1 = sigma_pinv @ np.ones(p_max)
x2 = sigma_pinv @ mu_hat
A = np.ones(p_max) @ x1
B = np.ones(p_max) @ x2
C = mu_hat @ x2
D = A * C - B**2
mu_grid = np.linspace(mu_hat.min(), mu_hat.max(), 200)
sigma_grid = np.sqrt(np.maximum((A * mu_grid**2 - 2 * B * mu_grid + C) / D, 0))
ax.plot(sigma_grid, mu_grid, color="black", linewidth=1.5, alpha=0.7, linestyle=":", label="Frontiere Sample (in-sample, pinv)")
sigma_in = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Sample")]["var_in"].values[0])
mean_in = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Sample")]["mean_in"].values[0]
sigma_out = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Sample")]["var_out"].values[0])
mean_out = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Sample")]["mean_out"].values[0]
ax.scatter([sigma_in], [mean_in], color="black", marker="*", s=200, zorder=5, label="Sample in-sample")
ax.scatter([sigma_out], [mean_out], color="black", marker="X", s=120, zorder=5, label="Sample out-of-sample")
ax.plot([sigma_in, sigma_out], [mean_in, mean_out], color="black", linestyle="--", linewidth=1, alpha=0.6)

# --- Clipping ---
sigma_c = resultats[p_max][0]["Clipping"]["sigma_cov"]
x1 = np.linalg.solve(sigma_c, np.ones(p_max))
x2 = np.linalg.solve(sigma_c, mu_hat)
A = np.ones(p_max) @ x1
B = np.ones(p_max) @ x2
C = mu_hat @ x2
D = A * C - B**2
sigma_grid = np.sqrt(np.maximum((A * mu_grid**2 - 2 * B * mu_grid + C) / D, 0))
ax.plot(sigma_grid, mu_grid, color="royalblue", linewidth=1.5, alpha=0.7, label="Frontiere Clipping (in-sample)")
sigma_in = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Clipping")]["var_in"].values[0])
mean_in = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Clipping")]["mean_in"].values[0]
sigma_out = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Clipping")]["var_out"].values[0])
mean_out = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Clipping")]["mean_out"].values[0]
ax.scatter([sigma_in], [mean_in], color="royalblue", marker="*", s=200, zorder=5, label="Clipping in-sample")
ax.scatter([sigma_out], [mean_out], color="royalblue", marker="X", s=120, zorder=5, label="Clipping out-of-sample")
ax.plot([sigma_in, sigma_out], [mean_in, mean_out], color="royalblue", linestyle="--", linewidth=1, alpha=0.6)

# --- Linear ---
sigma_c = resultats[p_max][0]["Linear"]["sigma_cov"]
x1 = np.linalg.solve(sigma_c, np.ones(p_max))
x2 = np.linalg.solve(sigma_c, mu_hat)
A = np.ones(p_max) @ x1
B = np.ones(p_max) @ x2
C = mu_hat @ x2
D = A * C - B**2
sigma_grid = np.sqrt(np.maximum((A * mu_grid**2 - 2 * B * mu_grid + C) / D, 0))
ax.plot(sigma_grid, mu_grid, color="forestgreen", linewidth=1.5, alpha=0.7, label="Frontiere Linear (in-sample)")
sigma_in = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Linear")]["var_in"].values[0])
mean_in = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Linear")]["mean_in"].values[0]
sigma_out = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Linear")]["var_out"].values[0])
mean_out = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="Linear")]["mean_out"].values[0]
ax.scatter([sigma_in], [mean_in], color="forestgreen", marker="*", s=200, zorder=5, label="Linear in-sample")
ax.scatter([sigma_out], [mean_out], color="forestgreen", marker="X", s=120, zorder=5, label="Linear out-of-sample")
ax.plot([sigma_in, sigma_out], [mean_in, mean_out], color="forestgreen", linestyle="--", linewidth=1, alpha=0.6)

# --- NLS ---
sigma_c = resultats[p_max][0]["NLS"]["sigma_cov"]
x1 = np.linalg.solve(sigma_c, np.ones(p_max))
x2 = np.linalg.solve(sigma_c, mu_hat)
A = np.ones(p_max) @ x1
B = np.ones(p_max) @ x2
C = mu_hat @ x2
D = A * C - B**2
sigma_grid = np.sqrt(np.maximum((A * mu_grid**2 - 2 * B * mu_grid + C) / D, 0))
ax.plot(sigma_grid, mu_grid, color="red", linewidth=1.5, alpha=0.7, label="Frontiere NLS (in-sample)")
sigma_in = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="NLS")]["var_in"].values[0])
mean_in = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="NLS")]["mean_in"].values[0]
sigma_out = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="NLS")]["var_out"].values[0])
mean_out = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="NLS")]["mean_out"].values[0]
ax.scatter([sigma_in], [mean_in], color="red", marker="*", s=200, zorder=5, label="NLS in-sample")
ax.scatter([sigma_out], [mean_out], color="red", marker="X", s=120, zorder=5, label="NLS out-of-sample")
ax.plot([sigma_in, sigma_out], [mean_in, mean_out], color="red", linestyle="--", linewidth=1, alpha=0.6)

# --- 1/N (pas de frontiere, juste les deux points) ---
sigma_in = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="1/N")]["var_in"].values[0])
mean_in = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="1/N")]["mean_in"].values[0]
sigma_out = np.sqrt(df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="1/N")]["var_out"].values[0])
mean_out = df_resultats[(df_resultats["p"]==p_max) & (df_resultats["tirage"]==0) & (df_resultats["methode"]=="1/N")]["mean_out"].values[0]
ax.scatter([sigma_in], [mean_in], color="gray", marker="*", s=200, zorder=5, label="1/N in-sample")
ax.scatter([sigma_out], [mean_out], color="gray", marker="X", s=120, zorder=5, label="1/N out-of-sample")
ax.plot([sigma_in, sigma_out], [mean_in, mean_out], color="gray", linestyle="--", linewidth=1, alpha=0.6)

ax.set_xlabel("Risque (ecart-type)")
# ax.set_xlim(0, 0.01)
ax.set_ylabel("Rendement moyen")
ax.set_title(f"Portefeuilles sur la frontiere efficiente, gamma={p_max/200:.2f} (Sample: frontiere via pinv, a interpreter avec prudence)")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()